# IFC Healing Semantic Relationships

In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 0. Import the Needed Classes and Libraries

In [ ]:
import os
import datetime

from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.TGraph import TGraph
from topologicpy.IFC import IFC
from topologicpy.Plotly import Plotly

## 1. Configuration

In [ ]:
# =============================================================================
# SECTION 1: CONFIGURATION
# =============================================================================

IFC_PATH = r"C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\IFC\Ifc2x3_Duplex_Architecture.ifc"
OUTPUT_BASE = r"C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\IFC"
_RUN_TS     = datetime.datetime.now().strftime("%Y%m%d_%H%M")
OUTPUT_DIR  = os.path.join(OUTPUT_BASE, "HEALED_IFC", _RUN_TS)

# Set your desired rendering parameters.
figure_width = 1200
figure_height = 1200
figure_background = "white"
renderer = "browser"

# Set your desired color and sizes for graph vertices and edges
graph_vertex_size = 10
graph_vertex_color = "red"

graph_edge_width=3
graph_edge_color="grey"

## 2. Create an IFCModel once

In [ ]:
print("=" * 72)
print(f"IFC path   : {IFC_PATH}")
print(f"Output dir : {OUTPUT_DIR}")
print("=" * 72)
ifc_model = IFC.ModelByPath(path=IFC_PATH)

## 3. Get IFC Spaces, Openings/Doors, Walls, Slabs, Stairs, and Ramps

In [ ]:


print("\nGetting IFC Entities...")
entities = IFC.Entities(ifc_model, silent=False)
print("IFC Entities:", len(entities))
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Getting Spaces...")
spaces = IFC.TopologiesByEntities(
    entities,
    includeTypes=["ifcspace"],
    dictionaryMode="full",
)
print(f"Spaces: {len(spaces)}")
# Convert to cells
print("Converting IfcSpaces to Cells...")
spaces = [Topology.SelfMerge(space) for space in spaces]

print("Getting Walls...")
walls = IFC.TopologiesByEntities(
    entities,
    includeTypes=["ifcwall", "ifcwallstandardcase"],
    dictionaryMode="basic"
)
print(f"Walls: {len(walls)}")

print("Getting Slabs...")
slabs = IFC.TopologiesByEntities(
    entities,
    includeTypes=["ifcslab"],
    dictionaryMode="basic"
)
print(f"Slabs: {len(slabs)}")

print("Getting Openings...")
doors = IFC.TopologiesByEntities(
    entities,
    includeTypes=["ifcopeningelement"],
    dictionaryMode="basic",
)
print(f"Openings: {len(doors)}")

print("Getting Stairs and ramps...")
stairs = IFC.TopologiesByEntities(
    entities,
    includeTypes=["ifcstair", "ifcramp", "ifcstairflight"],
    dictionaryMode="basic",
)
print(f"Stairs and Ramps: {len(stairs)}")

## 4. Build Semantic-Only Adjacency Graph

In [ ]:
print("\nBuilding TopologicPy Semantic Only Adjacency TGraph from IFC...")

semantic_adj_graph  = IFC.AdjacencyGraph(ifc_model,
                                      nodeTypes=["space"],
                                      connectingElementTypes=["wall", "slab"],
                                      viaConnectingElements=True,
                                      semanticConnections=True,
                                      spatialConnections=False,
                                      spatialRelationshipTypes = ["touches", "overlaps", "contains", "proximity"],
                                      proximityValues = [0.2, 0.5, 0.75],
                                      proximityLabels = ["near", "medium", "far"],
                                      useShortestDistance = True,
                                      dictionaryMode="Full",
                                      scale=1.0,
                                      circleSides=24,
                                      tolerance=0.01,
                                      silent=False
                                      )
print("Semantic Adjacency TGraph:", semantic_adj_graph)
if semantic_adj_graph is None:
    raise RuntimeError("IFC.AdjacencyGraph returned None. Check the IFC path, geometry, and TopologicPy IFC support.")

t_verts = TGraph.Vertices(semantic_adj_graph, copy=False)

key = "IFC_pset_Pset_SpaceCommon_Reference"

# Set Dictionary attributes and move the graph to the XY plane (Z=0)
for i, v in enumerate(t_verts):
        ifc_tag = str(v['dictionary'].get('IFC_long_name', "Unknown"))
        ifc_type = str(v['dictionary'].get('IFC_name', "0000"))
        display_label = ifc_type+"_"+ifc_tag
        v['dictionary']['display_label'] = display_label
        v['dictionary']["color"] = "red"
        v['dictionary']["size"] = 10

## 5. Visualise the Semantic Graph and the Building

In [ ]:
data1 = Plotly.DataByTopology(Cluster.ByTopologies(spaces+walls+doors+stairs), faceOpacity=0.1, showVertices=False)
data2 = Plotly.DataByTGraph(semantic_adj_graph,
                             vertexColor=graph_vertex_color,
                             vertexSize = graph_vertex_size,
                             edgeColor= "blue",
                             edgeWidth = graph_edge_width,
                             showVertexLabel=True,
                             vertexLabelFontSize=12,
                             vertexLabelKey="display_label")

fig = Plotly.FigureByData(data1+data2,
                          width=figure_width,
                          height=figure_height,
                          backgroundColor="white")

Plotly.Show(fig, renderer=renderer)

## 6. Build Spatial-Only Adjacency Graph

In [ ]:
print("\nBuilding TopologicPy Spatial Adjacency TGraph from IFC...")

spatial_adj_graph  = IFC.AdjacencyGraph(ifc_model,
                                        nodeTypes=["space"],
                                        connectingElementTypes=["slab"],
                                        viaConnectingElements=True,
                                        semanticConnections=False,
                                        spatialConnections=True,
                                        spatialRelationshipTypes = ["touches", "overlaps", "contains", "proximity"],
                                        proximityValues = [0.2, 0.5, 0.75],
                                        proximityLabels = ["near", "medium", "far"],
                                        useShortestDistance = True,
                                        dictionaryMode="Full",
                                        scale=1.0,
                                        circleSides=24,
                                        tolerance=0.01,
                                        silent=False
                                        )
print("Spatial Adjacency TGraph:", spatial_adj_graph)
if spatial_adj_graph is None:
    raise RuntimeError("IFC.AdjacencyGraph returned None. Check the IFC path, geometry, and TopologicPy IFC support.")

t_verts = TGraph.Vertices(spatial_adj_graph, copy=False)

key = "IFC_pset_Pset_SpaceCommon_Reference"

# Set Dictionary attributes and move the graph to the XY plane (Z=0)
for i, v in enumerate(t_verts):
        ifc_tag = str(v['dictionary'].get('IFC_long_name', "Unknown"))
        ifc_type = str(v['dictionary'].get('IFC_name', "0000"))
        display_label = ifc_type+"_"+ifc_tag
        v['dictionary']['display_label'] = display_label
        v['dictionary']["color"] = "red"
        v['dictionary']["size"] = 10



## 7. Visualise Semantic and Spatial Adjacency Graphs

In [ ]:
data3 = Plotly.DataByTGraph(spatial_adj_graph,
                             vertexColor=graph_vertex_color,
                             vertexSize = graph_vertex_size,
                             edgeColor= "orange",
                             edgeWidth = graph_edge_width,
                             showVertexLabel=True,
                             vertexLabelFontSize=12,
                             vertexLabelKey="display_label")

fig = Plotly.FigureByData(data1+data2+data3,
                          width=figure_width,
                          height=figure_height,
                          backgroundColor="white")

Plotly.Show(fig, renderer=renderer)

## 8. Create New Semantic Relationships in the IFC Model based on the Spatial Adjacency Graph

In [ ]:
result = IFC.SemanticRelationshipsByGraph(ifc_model, spatial_adj_graph)

## 9. Export the Semantically Healed IFC Model

In [ ]:
status = IFC.Export(ifc_model, path = OUTPUT_DIR, overwrite = True)